In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [14]:
basedf = pd.read_csv('/data/zsl23/preparation/basedf.csv', sep=",", header = 0, index_col=None)

In [15]:
basedf.head()

,#id,redshift,distance,galex.FUV,galex.FUV_err,galex.NUV,galex.NUV_err,sloan.sdss.u,sloan.sdss.u_err,sloan.sdss.g,...,herschel.pacs.green,herschel.pacs.green_err,herschel.pacs.red,herschel.pacs.red_err,herschel.spire.PSW,herschel.spire.PSW_err,herschel.spire.PMW,herschel.spire.PMW_err,herschel.spire.PLW,herschel.spire.PLW_err
0,ESO149-013,0,20.248498,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,49.629813,82.990714,7.899533,58.638459,7.301990,34.582998
1,NGC0007,0,20.606305,2.792007,0.130765,3.131622,0.109649,NaN,NaN,NaN,...,-585.993205,550.268549,479.281338,561.583315,419.146475,108.798696,343.955863,66.855994,168.743455,48.714171
2,ESO410-005,0,1.940886,0.169843,0.015266,0.505197,0.029494,NaN,NaN,NaN,...,606.719823,298.355089,575.689527,351.670292,49.891475,87.069055,15.661851,65.672827,9.789736,33.119280
3,IC0010,0,0.794328,NaN,NaN,82636.651201,332637.806656,NaN,NaN,NaN,...,216045.743398,37151.678713,214645.823529,118767.051682,110582.864955,59084.049307,52926.952540,40419.362730,21082.391740,14078.461687
4,NGC0115,0,28.707818,3.101272,0.142196,3.916178,0.110150,NaN,NaN,NaN,...,896.019859,600.566138,2157.135283,488.568231,912.792408,129.122311,495.308969,97.804529,259.551036,60.057107


In [16]:
# assigning columns/filters to wavelength ranges
galex_cols   = ['galex.FUV', 'galex.NUV'] # UV
sdss_cols    = ['sloan.sdss.u', 'sloan.sdss.g', 'sloan.sdss.r', 'sloan.sdss.i', 'sloan.sdss.z'] # VIS
nir_cols_tmass = ['2mass.J', '2mass.H', '2mass.Ks'] # NIR
mir_cols = ['wise.W1', 'wise.W2', 'wise.W3',
            'spitzer.irac.I1', 'spitzer.irac.I2', 'spitzer.irac.I3', 'spitzer.irac.I4'] # MIR
fir_cols_belowspire = ['wise.W4', 'spitzer.mips.24mu', 'spitzer.mips.70mu', 'spitzer.mips.160mu',
            'herschel.pacs.blue', 'herschel.pacs.green', 'herschel.pacs.red'] # near end of the FIR
spire_cols   = ['herschel.spire.PSW', 'herschel.spire.PMW', 'herschel.spire.PLW'] # far end of the FIR

In [17]:
# some functions to create criteria
def anypresent(cols):
    """Boolean mask: True where at least one of the given columns is not NaN."""
    return basedf[cols].notna().any(axis=1) # axis=1 looks at the columns. so for each row, any of the columns is not NaN, 
    # then the entire row is true.

def allpresent(cols):
    """Boolean mask: True only where all of the given columns is not NaN."""
    return basedf[cols].notna().all(axis=1) 
    
# pandas.notna() identifies non-missing values in a Series/DataFrame. It returns a Boolean mask (True for present values, False for NaNs).

def count_present(cols):
    """Row count of non NaN columns among the given list."""
    return basedf[cols].notna().sum(axis=1) # for each row, sum up the number of columns that are true.

In [18]:
# criteria
crit_galex = anypresent(galex_cols)
crit_sdss  = allpresent(sdss_cols)
crit_nir = allpresent(nir_cols_tmass)

crit_mir = count_present(mir_cols) >= 3
crit_fir = count_present(fir_cols_belowspire) >= 3
crit_spire = allpresent(spire_cols)

# combine the criteriaglobalgaldf[value_cols] = globalgaldf[value_cols] * 1000  # convert Jy -> mJy
final_mask = (crit_galex & crit_sdss & crit_nir & crit_mir & crit_fir & crit_spire)

In [19]:
print(f"  GALEX (FUV or NUV):              {crit_galex.sum()}")
print(f"  SDSS (all 5 bands):              {crit_sdss.sum()}")
print(f"  NIR (all 3 bands):               {crit_nir.sum()}")
print(f"  MIR >=3 bands:                   {crit_mir.sum()}")
print(f"  FIR >=3 bands:                   {crit_fir.sum()}")
print(f"  SPIRE (all 3 bands):             {crit_spire.sum()}")
print()
print(f"FINAL count (all criteria met): {final_mask.sum()}")

  GALEX (FUV or NUV):              748
  SDSS (all 5 bands):              578
  NIR (all 3 bands):               768
  MIR >=3 bands:                   777
  FIR >=3 bands:                   623
  SPIRE (all 3 bands):             756

FINAL count (all criteria met): 417


In [20]:
basegal = basedf[final_mask]
print(len(basegal))

417


In [21]:
basegalnan = basegal.fillna('NaN')
basegalnan

,#id,redshift,distance,galex.FUV,galex.FUV_err,galex.NUV,galex.NUV_err,sloan.sdss.u,sloan.sdss.u_err,sloan.sdss.g,...,herschel.pacs.green,herschel.pacs.green_err,herschel.pacs.red,herschel.pacs.red_err,herschel.spire.PSW,herschel.spire.PSW_err,herschel.spire.PMW,herschel.spire.PMW_err,herschel.spire.PLW,herschel.spire.PLW_err
30,NGC0584,0,19.952632,0.620234,0.088756,2.073565,0.142392,35.421674,3.026211,171.536679,...,5691.217333,1890.06276,2893.126455,916.770993,783.376931,365.780969,353.564684,294.936519,102.643637,109.527806
31,NGC0586,0,24.931731,0.23176,0.028581,0.509059,0.05409,3.926040,1.509732,15.139204,...,837.131088,516.396852,1596.123301,199.816541,933.594589,112.204362,430.023547,83.570652,143.357723,44.762027
35,NGC0678,0,27.870000,0.543844,0.790122,0.862554,0.162827,4.248485,5.890089,37.589727,...,1223.855665,287.974411,3233.413256,337.464978,2164.558665,303.439768,1012.796269,168.511282,293.963453,99.456850
38,NGC0855,0,9.638295,1.438534,0.077612,2.963126,0.118233,10.961062,2.839492,32.987062,...,2728.886163,678.631371,2102.084655,469.056795,1095.840755,229.814847,473.828802,119.802497,147.406547,75.217007
39,PGC008962,0,30.502458,0.63599,0.035677,0.807483,0.04207,0.994889,0.260776,2.818215,...,NaN,NaN,45.864191,397.613791,64.433866,93.869486,30.577573,69.297480,-0.249728,44.859224
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
736,NGC5750,0,29.700000,1.306674,0.067534,2.394977,0.087929,15.891191,0.731238,63.874835,...,5712.996337,1304.909886,4921.182838,963.722097,2560.522564,214.398484,1125.747282,140.398487,379.102596,67.451531
738,NGC5866,0,14.454393,1.114304,0.248975,4.893241,0.307608,63.545141,14.709548,274.734254,...,11473.029689,3865.679533,17358.682561,2580.94796,7980.742536,629.859582,3641.236554,414.873905,1042.790937,241.656376
741,NGC5907,0,17.218688,8.612285,1.140696,12.844704,0.706781,48.089479,12.183771,177.618041,...,63167.768837,4464.305649,93321.702245,6551.386773,55649.163036,3062.253504,25814.080111,1423.110606,10190.034970,562.535723
742,NGC6143,0,78.577280,1.308862,0.062217,2.040994,0.064979,4.634019,0.800343,14.337235,...,1729.471115,615.60095,3338.015143,678.670789,1882.966981,122.377490,866.997663,79.070354,315.264443,44.763273


In [25]:
# there is one problematic galaxy PGC028759, need to remove it from analysis
mask = np.column_stack([basegalnan['#id'].str.contains(r"PGC028759", na=False) for col in basegalnan])
basegalnan = basegalnan[~mask]

In [26]:
basegalnan.to_csv('basegalmJynan.txt' ,sep=' ', index=False) # to be used in CIGALE

In [33]:
mask = np.column_stack([basegalnan['#id'].str.contains(r"PGC028759", na=False) for col in basegalnan])
basegalnan.loc[mask.any(axis = 1)] # any column (in this case, id) that is true

,#id,redshift,distance,galex.FUV,galex.FUV_err,galex.NUV,galex.NUV_err,sloan.sdss.u,sloan.sdss.u_err,sloan.sdss.g,...,herschel.pacs.green,herschel.pacs.green_err,herschel.pacs.red,herschel.pacs.red_err,herschel.spire.PSW,herschel.spire.PSW_err,herschel.spire.PMW,herschel.spire.PMW_err,herschel.spire.PLW,herschel.spire.PLW_err
